# MILK10k — Multimodal Baseline (late fusion)

Bu notebook ilk referans noktasını kurar. Amaç en iyi skoru almak değil, sonraki denemeleri kıyaslayabileceğimiz basit bir başlangıç noktası oluşturmak.

| Parça | Girdi | Model |
|---|---|---|
| Görüntü modeli | klinik + dermoskopik görüntü → dondurulmuş DINOv2-large → 2 × 2048 özellik | Logistic Regression |
| Tabular model | yaş, cinsiyet, cilt tonu, lezyon yeri + 14 MONET skoru | LightGBM |
| Fusion | iki modelin olasılıklarının ortalaması | – |

**Değerlendirme:** lezyon bazında 5-fold stratified cross-validation. Ana metrik, yarışmanın resmi metriği olan **macro-F1**: her sınıfın F1'inin düz ortalaması. Yani 9 örnekli MAL_OTH da 2522 örnekli BCC kadar önemli.

Veri yükleme, tabular özellikler, iki model ve fold'lar `02_finetune` ile ortak; tek kopyaları `notebooks/common.py`'de.

Sonda modeller tüm train verisiyle yeniden eğitilip test seti için `outputs/baseline/submission.csv` yazılır.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.models.feature_extraction import create_feature_extractor
from tqdm import tqdm
from transformers import AutoModel

pd.set_option("display.max_columns", 50)

# proje kökü: içinde data/ klasörü olan ilk üst klasör (notebook hangi klasörden açılırsa açılsın çalışsın diye)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA = ROOT / "data"
CACHE = ROOT / "outputs" / "cache"
OUT = ROOT / "outputs" / "baseline"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "notebooks"))
# 02_finetune ile ortak kod: notebooks/common.py
from common import (CLASSES, image_model, lesion_ids, load_labels, load_split, make_folds, oof_proba,
                    tabular_features, tabular_model)

C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Veriyi yükleme

Metadata'da her lezyonun iki satırı var (klinik + dermoskopik görüntü). `load_split` (`common.py`) bunları **lezyon başına tek satıra** indiriyor: hasta bilgileri + iki görüntünün MONET skorları (`clin_…`, `derm_…`) + iki görüntünün dosya yolu (`clin`, `derm` sütunları).

In [2]:
train, test = load_split(DATA, "Training"), load_split(DATA, "Test")
print(f"train: {len(train)} lezyon | test: {len(test)} lezyon")
train.head(3)

train: 5240 lezyon | test: 479 lezyon


,age_approx,sex,skin_tone_class,site,clin_MONET_ulceration_crust,derm_MONET_ulceration_crust,clin_MONET_hair,derm_MONET_hair,clin_MONET_vasculature_vessels,derm_MONET_vasculature_vessels,clin_MONET_erythema,derm_MONET_erythema,clin_MONET_pigmented,derm_MONET_pigmented,clin_MONET_gel_water_drop_fluid_dermoscopy_liquid,derm_MONET_gel_water_drop_fluid_dermoscopy_liquid,clin_MONET_skin_markings_pen_ink_purple_pen,derm_MONET_skin_markings_pen_ink_purple_pen,clin,derm
lesion_id,,,,,,,,,,,,,,,,,,,,
IL_0000652,70.0,male,1,head_neck_face,0.166749,0.659859,0.163601,0.156478,0.002284,0.016397,0.124315,0.032357,0.719495,0.847014,0.220399,0.138121,0.237601,0.148776,D:\Vcode_4\test\main_code\data\MILK10k_Trainin...,D:\Vcode_4\test\main_code\data\MILK10k_Trainin...
IL_0003176,45.0,female,5,head_neck_face,0.348609,0.392950,0.614718,0.897668,0.013415,0.367882,0.447485,0.645776,0.061977,0.122108,0.296341,0.719937,0.058006,0.329812,D:\Vcode_4\test\main_code\data\MILK10k_Trainin...,D:\Vcode_4\test\main_code\data\MILK10k_Trainin...
IL_0004688,50.0,male,3,lower_extremity,0.889925,0.548515,0.120788,0.208269,0.004546,0.196453,0.487298,0.519808,0.036550,0.058424,0.146776,0.319812,0.086027,0.262883,D:\Vcode_4\test\main_code\data\MILK10k_Trainin...,D:\Vcode_4\test\main_code\data\MILK10k_Trainin...


### Etiketler

GroundTruth dosyası one-hot; her lezyonu tek bir sınıf indeksine (0–10) çeviriyoruz. Dağılım çok dengesiz: BCC %48, MAL_OTH sadece 9 lezyon.

In [3]:
y = load_labels(DATA, train.index)  # one-hot -> sınıf indeksi

counts = np.bincount(y)
pd.DataFrame({"n": counts, "%": (counts / len(y) * 100).round(1)}, index=CLASSES)

,n,%
AKIEC,303,5.8
BCC,2522,48.1
BEN_OTH,44,0.8
BKL,544,10.4
DF,52,1.0
INF,50,1.0
MAL_OTH,9,0.2
MEL,450,8.6
NV,746,14.2
SCCKA,473,9.0


## 2. Tabular özellikler

Lezyon başına 18 özellik (`tabular_features`, `common.py`): yaş, cinsiyet, cilt tonu, lezyon yeri + 7 MONET skoru × 2 görüntü. Cinsiyet ve lezyon yeri kategorik. Eksik değerlere (yaş, lezyon yeri) dokunmuyoruz, LightGBM bunları kendisi işliyor.

In [4]:
sites = sorted(train["site"].dropna().unique())
X_tab, X_tab_test = tabular_features(train, sites), tabular_features(test, sites)
print("train:", X_tab.shape, "| test:", X_tab_test.shape)

train: (5240, 18) | test: (479, 18)


## 3. Görüntü özellikleri (DINOv2-large)

Görüntüleri **eğitmeden** (dondurulmuş) bir özellik çıkarıcıdan geçiriyoruz. Şu an kullanılan model **DINOv2-large**: etiketsiz çok sayıda görüntüyle, kendi kendine öğrenme (self-supervised) yöntemiyle eğitilmiş bir Vision Transformer. Her görüntüden son katmandaki CLS token'ı (1024) ve patch token'larının ortalaması (1024) alınıyor; klinik + dermoskopik birlikte lezyon başına **4096 özellik**.

`BACKBONE` ile ResNet'e de geçilebilir. O durumda ResNet'in üç ara katmanının (layer2, layer3, layer4) çıkışları kullanılıyor (lezyon başına 7168 özellik). Kıyas için `history.md`: ResNet101 0.452, DINOv2-large 0.487.

Bu hesaplama bir kez yapılıp `outputs/cache/` altına kaydediliyor (GPU ile birkaç dakika). Cache varsa doğrudan oradan okunuyor; yeniden hesaplatmak için oradaki `.npz` dosyalarını silmek yeterli.

In [5]:
class ImageDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths, self.transform = paths, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert("RGB"))


BACKBONE = "dinov2_large"  # "dinov2_large" | "resnet101" | "resnet50"
RESNETS = {
    "resnet50": (models.resnet50, models.ResNet50_Weights.IMAGENET1K_V1),
    "resnet101": (models.resnet101, models.ResNet101_Weights.IMAGENET1K_V1),
}
LAYERS = ["layer2", "layer3", "layer4"]  # ResNet'te özellikleri alınan katmanlar
DINO_TRANSFORM = T.Compose([T.Resize(256, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(),
                            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


@torch.no_grad()
def cnn_features(paths):
    """Görüntü başına bir özellik vektörü.
    DINOv2: son katmandaki CLS token + patch token'ların ortalaması (1024 + 1024).
    ResNet: LAYERS'daki katmanların çıkışı, global average pooling ile (512 + 1024 + 2048)."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if BACKBONE == "dinov2_large":
        model, transform = AutoModel.from_pretrained("facebook/dinov2-large"), DINO_TRANSFORM
        embed = lambda out: torch.cat([out.pooler_output, out.last_hidden_state[:, 1:].mean(1)], 1)
    else:
        arch, weights = RESNETS[BACKBONE]
        model = create_feature_extractor(arch(weights=weights), return_nodes={l: l for l in LAYERS})
        transform = weights.transforms()
        embed = lambda out: torch.cat([out[l].mean(dim=(2, 3)) for l in LAYERS], 1)
    model.eval().to(device)
    # num_workers=0: Windows'ta notebook içinde tanımlanan Dataset çoklu worker ile çalışmaz
    loader = DataLoader(ImageDataset(paths, transform), batch_size=32, num_workers=0)
    return np.concatenate([embed(model(x.to(device))).cpu().numpy() for x in tqdm(loader, desc=BACKBONE)])


def image_features(df, split):
    """Lezyon başına [klinik | dermoskopik] özellik vektörü. İlk seferde hesaplanıp cache'lenir."""
    name = BACKBONE if BACKBONE.startswith("dinov2") else f"{BACKBONE}_layers"
    cache = CACHE / f"{name}_{split}.npz"
    ids = lesion_ids(df)
    if cache.exists():
        z = np.load(cache)
        assert (z["lesion_id"] == ids).all(), f"cache eski, {cache.name} dosyasını silip tekrar çalıştırın"
        return z["X"]
    n = len(df)
    F = cnn_features(df["clin"].tolist() + df["derm"].tolist())
    X = np.hstack([F[:n], F[n:]])
    CACHE.mkdir(parents=True, exist_ok=True)
    np.savez(cache, X=X, lesion_id=ids)
    return X


X_img, X_img_test = image_features(train, "train"), image_features(test, "test")
print("train:", X_img.shape, "| test:", X_img_test.shape)

train: (5240, 4096) | test: (479, 4096)


dinov2_large: 100%|██████████| 30/30 [00:14<00:00,  2.08it/s]


## 4. Modeller ve cross-validation

Model tanımları (`image_model`, `tabular_model`), fold'lar (`make_folds`) ve out-of-fold tahmin (`oof_proba`) `common.py`'de; `02_finetune` aynılarını kullanıyor.

- İki modelde de `class_weight="balanced"`: nadir sınıflara daha fazla ağırlık veriyor, macro-F1 için önemli.
- Görüntü modelinde güçlü regularization (küçük `C`) şart, yoksa 4096 özellik ~4200 eğitim örneğini ezberliyor. `C=0.03`, DINOv2-large özelliklerinde 0.001–0.1 arasından fusion macro-F1'e göre seçildi. (ResNet101 katman özelliklerinde en iyisi 0.001'di; `BACKBONE` değiştirilirse C de değiştirilmeli.) Aynı CV üzerinde seçildiği için skor hafif iyimser olabilir.
- İki model de aynı 5 fold'u kullanıyor. Her lezyonun tahmini, o lezyonu eğitimde hiç görmemiş modelden geliyor (*out-of-fold* tahmin).
- Tahmin = en olası sınıf. Bu durumda resmi metrik (her sınıf ≥0.5 eşiğiyle ikiliye çevrilip macro-F1) sklearn'ün macro-F1'i ile birebir aynı.

In [6]:
def scores(y, proba, folds):
    pred = proba.argmax(1)
    fold_f1 = [f1_score(y[va], pred[va], average="macro", zero_division=0) for _, va in folds]
    return {"macro_f1": f1_score(y, pred, average="macro", zero_division=0),
            "macro_f1_fold_std": np.std(fold_f1),  # farkların gürültü mü gerçek mi olduğunu anlamak için
            "balanced_acc": balanced_accuracy_score(y, pred),
            "macro_auc": roc_auc_score(y, proba, multi_class="ovr"),
            "accuracy": accuracy_score(y, pred)}

In [7]:
%%time
folds = make_folds(y)
p_tab = oof_proba(tabular_model, X_tab, y, folds)
p_img = oof_proba(image_model, X_img, y, folds)

CPU times: total: 1min 20s
Wall time: 20.2 s


## 5. Fusion ve sonuçlar

Late fusion: `p = FUSION_W · p_img + (1 − FUSION_W) · p_tab`. DINOv2 ile ağırlık ve C iç içe CV ile birlikte tarandı: ayarlı ağırlık 0.488, düz ortalama 0.487 → fark yok, düz ortalama (`FUSION_W = 0.5`) kullanılıyor (`history.md` #12).

`macro_f1_fold_std` fold'lar arası sapma. İleride iki deneme arasındaki fark bunun ~2 katından küçükse gürültü olabilir.

In [8]:
FUSION_W = 0.5  # görüntü modelinin ağırlığı (0.5 = düz ortalama)
p_fus = FUSION_W * p_img + (1 - FUSION_W) * p_tab

preds = {"tabular (LightGBM)": p_tab, f"image ({BACKBONE} + LR)": p_img, f"fusion (w={FUSION_W})": p_fus}
results = pd.DataFrame({name: scores(y, p, folds) for name, p in preds.items()}).T.round(4)
results.to_csv(OUT / "cv_results.csv")
results

,macro_f1,macro_f1_fold_std,balanced_acc,macro_auc,accuracy
tabular (LightGBM),0.3334,0.0134,0.3265,0.8423,0.6120
image (dinov2_large + LR),0.4821,0.0090,0.4732,0.9080,0.7082
fusion (w=0.5),0.4867,0.0241,0.4615,0.9099,0.7273


### Sınıf bazında F1

In [9]:
per_class = pd.DataFrame({name: f1_score(y, p.argmax(1), average=None, labels=range(len(CLASSES)), zero_division=0)
                          for name, p in preds.items()}, index=CLASSES).round(3)
per_class.insert(0, "n", np.bincount(y, minlength=len(CLASSES)))
per_class.to_csv(OUT / "cv_per_class_f1.csv")
per_class

,n,tabular (LightGBM),image (dinov2_large + LR),fusion (w=0.5)
AKIEC,303,0.231,0.418,0.425
BCC,2522,0.757,0.849,0.858
BEN_OTH,44,0.105,0.219,0.254
BKL,544,0.270,0.446,0.451
DF,52,0.030,0.505,0.506
INF,50,0.110,0.256,0.152
MAL_OTH,9,0.000,0.000,0.000
MEL,450,0.522,0.577,0.591
NV,746,0.721,0.747,0.774
SCCKA,473,0.499,0.650,0.652


### Confusion matrix (fusion)

Satırlar gerçek sınıf, sütunlar tahmin edilen sınıf; köşegen doğru tahminler.

In [10]:
cm = pd.DataFrame(confusion_matrix(y, p_fus.argmax(1)), index=CLASSES, columns=CLASSES)
cm.to_csv(OUT / "cv_confusion_matrix.csv")
cm

,AKIEC,BCC,BEN_OTH,BKL,DF,INF,MAL_OTH,MEL,NV,SCCKA,VASC
AKIEC,121,78,2,49,0,0,0,5,2,46,0
BCC,42,2213,2,91,1,11,0,36,22,102,2
BEN_OTH,2,10,8,6,1,2,0,5,8,1,1
BKL,42,152,2,223,1,1,0,54,26,43,0
DF,0,10,0,3,20,4,0,4,8,3,0
INF,5,22,0,1,1,6,0,1,11,3,0
MAL_OTH,0,5,0,3,0,0,0,1,0,0,0
MEL,9,31,1,21,1,1,0,269,115,0,2
NV,3,25,2,26,1,3,0,85,596,2,3
SCCKA,42,83,1,20,0,1,0,0,0,326,0


## 6. Test tahmini ve submission

Modeller bu kez tüm train verisiyle eğitilip test lezyonları tahmin ediliyor. Resmi skorlama her sınıfı ≥0.5 eşiğiyle ikiliye çevirdiği için submission'da en olası sınıfa 1, diğerlerine 0 yazılıyor. Ham olasılıklar ayrıca `test_probabilities.csv` dosyasında.

In [11]:
img_model = image_model().fit(X_img, y)
tab_model = tabular_model().fit(X_tab, y)
p_test = FUSION_W * img_model.predict_proba(X_img_test) + (1 - FUSION_W) * tab_model.predict_proba(X_tab_test)

# ilk sütun 'lesion_id' olmalı: yarışma sayfası 'lesion' diyor ama skorlama kodu 'lesion_id' istiyor
sub = pd.DataFrame(np.eye(len(CLASSES))[p_test.argmax(1)], columns=CLASSES, index=test.index).rename_axis("lesion_id")
sub.to_csv(OUT / "submission.csv")
pd.DataFrame(p_test, columns=CLASSES, index=test.index).rename_axis("lesion_id").to_csv(OUT / "test_probabilities.csv")

# test tahminlerinin sınıf dağılımı, train'deki gerçek dağılımla yan yana
pd.DataFrame({"train gerçek %": counts / len(y) * 100,
              "test tahmin %": sub.sum().values / len(sub) * 100}, index=CLASSES).round(1)

,train gerçek %,test tahmin %
AKIEC,5.8,8.4
BCC,48.1,29.9
BEN_OTH,0.8,0.4
BKL,10.4,14.8
DF,1.0,0.6
INF,1.0,1.7
MAL_OTH,0.2,0.0
MEL,8.6,6.9
NV,14.2,11.9
SCCKA,9.0,24.8


## Notlar

- **Referans skor:** fusion CV macro-F1 ≈ 0.487 (fold sapması ≈ 0.024). Yeni denemeleri bununla kıyaslıyoruz.
- Fusion iki modelden de iyi, ama DINOv2 ile tabular modelin katkısı küçüldü (görüntü tek başına 0.484, fusion 0.487).
- MAL_OTH (9 örnek) hiç tahmin edilmiyor; macro-F1'de bu sınıfın F1'i 0 sayılıyor.
- Test tahminlerinde SCCKA oranı train'dekinin yaklaşık 3 katı ve iki model de bunu bağımsız olarak gösteriyor → test setinin sınıf dağılımı muhtemelen farklı, leaderboard skoru CV'den sapabilir.
- Sıradaki en büyük kazanç muhtemelen CNN'i fine-tune etmek (şu an donmuş DINOv2 özellikleri kullanılıyor).

## Deney geçmişi

Her denemede yalnızca bir şey değişiyor; fold'lar ve diğer tüm ayarlar sabit. Ayrıntılı kayıt: proje kökündeki `history.md`.

| # | Değişiklik | Görüntü macro-F1 | Fusion macro-F1 | Sonuç |
|---|---|---|---|---|
| 1 | Baseline: ResNet50 + LR (C=0.003) | 0.388 | 0.425 | referans |
| 2 | ResNet50 → **ResNet101** | 0.395 | **0.436** | fusion 5 fold'un 5'inde daha iyi (ort. +0.010) → ResNet101 ile devam |
| 3 | LR yerine SVM (RBF), MLP, LightGBM | 0.335 – 0.381 | 0.343 – 0.411 | hiçbiri LR'yi tek bir fold'da bile geçemedi → LR ile devam |
| 4 | class_weight: balanced yerine ağırlıksız (None) ve sqrt | 0.376 – 0.404 | 0.351 – 0.419 | ağırlıksız fusion 0.351'e düştü; 9 kombinasyondan hiçbiri balanced/balanced'ı tek fold'da bile geçemedi → balanced ile devam |
| 5 | Sadece son katman yerine ResNet101'in **layer2 + layer3 + layer4** çıkışları (GAP), C=0.001 | 0.427 | **0.452** | 5 fold'un 5'inde daha iyi (ort. +0.018) → çok katmanlı ile devam. Tek başına layer2 / layer3 ve layer3+layer4 kazandırmadı. 12 kombinasyon arasından seçildiği için hafif iyimser olabilir; C=0.003'te de +0.013 (4/5 fold) |
| 6 | Fusion yöntemi: ağırlıklı ortalama, geometrik ortalama, stacking (iç içe CV ile) | – | 0.371 – 0.456 | ağırlıklı ortalama (w=0.55) +0.005, 3/5 fold → gürültü seviyesinde; geometrik ortalama ve stacking daha kötü → düz ortalama ile devam |
| 7 | Görüntü özellikleri: ResNet101 yerine **DINOv2-large** (CLS + patch ortalaması), C=0.03 | 0.484 | **0.487** | +0.034, 5 fold'un 5'inde daha iyi. DINOv2-base 0.467, MONET (dermatoloji CLIP) 0.457, ResNet+DINOv2 birlikte 0.484 → #12'de notebook'a alındı |
| 8 | Kırpmasız giriş (224×299, 336×448, 450×600) | 0.400 – 0.425 | 0.425 – 0.437 | üç çözünürlükte de daha kötü (−0.017…−0.028) → ortadan kırpma kalıyor |
| 9 | TTA: 8 görünüm (4 rotasyon × ayna) özellik ortalaması, C=0.01 | 0.451 | 0.462 | +0.009, 4/5 fold → küçük ama tutarlı kazanç |
| 10 | Karar eşikleri: sınıf başına log-bias (iç içe CV) | – | 0.438 | −0.016, 1/5 fold. Aynı veride ölçülse 0.472 görünüyordu → nadir sınıflarda ezber, kullanılmıyor |
| 11 | Tabular: MONET derm−klinik farkı (+ ortalaması) | – | 0.432 – 0.435 | −0.018, tabular model de geriledi → kullanılmıyor |
| 12 | DINOv2-large notebook'a alındı; fusion ağırlığı + C iç içe CV ile birlikte tarandı | 0.484 | **0.487** | ayarlı (C, w) 0.488 → düz ortalamadan farkı yok (aynı veride ölçülse 0.494 görünüyordu); C=0.03 + düz ortalama ile devam |
| 13 | **Fine-tuning** (`02_finetune.ipynb`): DINOv2-base + donmuş DINOv2-large + tabular (1/3'er) | 0.527 | **0.544** | +0.052, 5/5 fold; test (leaderboard) 0.421 → **0.511** |